# 1. Project Overview

# Machine Learning Analysis of Educational Outcomes

## Case Study: Australian Council for Educational Research (ACER)

### Project Aim

This project investigates how student characteristics, engagement,
attendance and academic information can be used to understand and
predict educational outcomes.

Two publicly available education datasets are analysed using two
machine learning algorithms: Random Forest and K-Nearest Neighbours (KNN).

The findings will be compared to determine whether the two datasets
provide consistent, complementary or contradictory insights into
student educational outcomes.

In [1]:
# Import libraries required for data manipulation, visualisation,
# preprocessing, machine learning models, and model evaluation.import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 2. Dataset 1 — OpenML 43415

### Loading the OpenML Dataset

The OpenML dataset was provided in ARFF format. An initial attempt was
made to load the file using SciPy's ARFF reader. This method could not
process the string attributes in the dataset, so an alternative ARFF
reader was used.

In [2]:
# Install the ARFF reader required to load the OpenML dataset.
!pip install liac-arff

## 2.1 Data Loading and Initial Inspection

In [3]:
# Load the OpenML dataset from the ARFF file and convert it
# into a pandas DataFrame for analysis.
import arff
import pandas as pd

with open("dataset", "r") as f:
    data = arff.load(f)

df1 = pd.DataFrame(data["data"], columns=[x[0] for x in data["attributes"]])

df1.head()

,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,VisITedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,15,16,2,20,Yes,Good,Under-7,M
1,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,20,20,3,25,Yes,Good,Under-7,M
2,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,10,7,0,30,No,Bad,Above-7,L
3,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,30,25,5,35,No,Bad,Above-7,L
4,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,40,50,12,50,No,Bad,Above-7,M


In [4]:
# Check the number of observations and attributes in the dataset.
df1.shape

(480, 17)

In [5]:
# Display the names of all dataset attributes.
df1.columns.tolist()

['gender',
 'NationalITy',
 'PlaceofBirth',
 'StageID',
 'GradeID',
 'SectionID',
 'Topic',
 'Semester',
 'Relation',
 'raisedhands',
 'VisITedResources',
 'AnnouncementsView',
 'Discussion',
 'ParentAnsweringSurvey',
 'ParentschoolSatisfaction',
 'StudentAbsenceDays',
 'Class']

In [6]:
# Examine the number of observations in each outcome class.
df1["Class"].value_counts()

Class
M    211
H    142
L    127
Name: count, dtype: int64

In [7]:
# Calculate the percentage distribution of observations across classes.
df1["Class"].value_counts(normalize=True) * 100

Class
M    43.958333
H    29.583333
L    26.458333
Name: proportion, dtype: float64

In [8]:
# Check for missing values in each attribute.
df1.isnull().sum()

gender                      0
NationalITy                 0
PlaceofBirth                0
StageID                     0
GradeID                     0
SectionID                   0
Topic                       0
Semester                    0
Relation                    0
raisedhands                 0
VisITedResources            0
AnnouncementsView           0
Discussion                  0
ParentAnsweringSurvey       0
ParentschoolSatisfaction    0
StudentAbsenceDays          0
Class                       0
dtype: int64

In [9]:
# Inspect the data types of the dataset attributes.
df1.dtypes

gender                      object
NationalITy                 object
PlaceofBirth                object
StageID                     object
GradeID                     object
SectionID                   object
Topic                       object
Semester                    object
Relation                    object
raisedhands                  int64
VisITedResources             int64
AnnouncementsView            int64
Discussion                   int64
ParentAnsweringSurvey       object
ParentschoolSatisfaction    object
StudentAbsenceDays          object
Class                       object
dtype: object

In [10]:
# Identify categorical attributes and inspect their unique values
# before preprocessing.categorical_cols = df1.select_dtypes(include="object").columns

for col in categorical_cols:
    print(f"\n{col}:")
    print(df1[col].unique())

NameError: name 'categorical_cols' is not defined

## 2.2 Data Preparation

In [ ]:
# Separate the predictor variables from the target class.
X = df1.drop(columns=["Class"])
y = df1["Class"]

In [ ]:
# Check the dimensions of the predictor and target data.
X.shape, y.shape

In [ ]:
# Separate categorical and numerical predictors for preprocessing.
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("\nNumerical columns:", numerical_cols)

## 2.3 Train-Test Split and Preprocessing

In [ ]:
# Split the data into training and testing sets while preserving
# the class distribution through stratification.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Confirm the number of observations in the training and testing sets.
print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

In [ ]:
# Encode categorical variables and standardise numerical variables
# before applying the machine learning models.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numerical", StandardScaler(), numerical_cols)
    ]
)

## 2.4 Random Forest Classification

In [ ]:
# Create a pipeline that preprocesses the data and trains
# a Random Forest classifier.
rf_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ))
    ]
)

rf_classifier.fit(X_train, y_train)

In [ ]:
# Generate class predictions for the test set using Random Forest.
y_pred_rf = rf_classifier.predict(X_test)

In [ ]:
# Display a sample of the Random Forest predictions.
print("Random Forest predictions:")
print(y_pred_rf[:10])

In [ ]:
# Calculate classification metrics to evaluate the Random Forest model.
rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_precision = precision_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

rf_recall = recall_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

rf_f1 = f1_score(
    y_test,
    y_pred_rf,
    average="weighted"
)

print("Random Forest Results")
print("---------------------")
print(f"Accuracy : {rf_accuracy:.4f}")
print(f"Precision: {rf_precision:.4f}")
print(f"Recall   : {rf_recall:.4f}")
print(f"F1-score : {rf_f1:.4f}")

In [ ]:
# Generate a confusion matrix to examine the Random Forest
# classification results across the three outcome classes.cm_rf = confusion_matrix(y_test, y_pred_rf, labels=["L", "M", "H"])

print("Confusion Matrix")
print(cm_rf)

## 2.5 KNN Classification

In [ ]:
# Create a pipeline that preprocesses the data and applies
# K-Nearest Neighbours classification.
knn_classifier = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]
)

knn_classifier.fit(X_train, y_train)

In [ ]:
# Generate class predictions for the test set using KNN.
y_pred_knn = knn_classifier.predict(X_test)

In [ ]:
# Calculate classification metrics to evaluate the KNN model.
knn_accuracy = accuracy_score(y_test, y_pred_knn)

knn_precision = precision_score(
    y_test,
    y_pred_knn,
    average="weighted"
)

knn_recall = recall_score(
    y_test,
    y_pred_knn,
    average="weighted"
)

knn_f1 = f1_score(
    y_test,
    y_pred_knn,
    average="weighted"
)

print("KNN Results")
print("-----------")
print(f"Accuracy : {knn_accuracy:.4f}")
print(f"Precision: {knn_precision:.4f}")
print(f"Recall   : {knn_recall:.4f}")
print(f"F1-score : {knn_f1:.4f}")

In [ ]:
# Generate a confusion matrix for the KNN classification results.
cm_knn = confusion_matrix(y_test, y_pred_knn, labels=["L", "M", "H"])

print("KNN Confusion Matrix")
print(cm_knn)

## 2.6 Model Comparison

In [ ]:
# Compare Random Forest and KNN using the classification
# evaluation metrics.
model_comparison1 = pd.DataFrame({
    "Model": ["Random Forest", "KNN"],
    "Accuracy": [rf_accuracy, knn_accuracy],
    "Precision": [rf_precision, knn_precision],
    "Recall": [rf_recall, knn_recall],
    "F1-score": [rf_f1, knn_f1]
})

model_comparison1

## 2.7 Feature Importance

In [ ]:
# Extract the processed feature names and their Random Forest
# importance scores, then rank the features by importance.feature_names1 = (
    rf_classifier.named_steps["preprocessor"]
    .get_feature_names_out()
)

importances1 = rf_classifier.named_steps["model"].feature_importances_

feature_importance1 = pd.DataFrame({
    "Feature": feature_names1,
    "Importance": importances1
}).sort_values("Importance", ascending=False)

feature_importance1.head(15)

In [ ]:
# Visualise the ten most important features identified by
# the Random Forest model.
top10_1 = feature_importance1.head(10).sort_values("Importance")

plt.figure(figsize=(10, 6))
plt.barh(top10_1["Feature"], top10_1["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 10 Predictive Features — OpenML 43415")
plt.tight_layout()
plt.show()

## 3.Dataset 2 - UCI Student Performance

The second dataset is the UCI Student Performance dataset. This analysis
uses the Mathematics dataset to investigate whether student demographic,
social, family and educational characteristics can be used to predict
final mathematics performance.

## 3.1 Data Loading and Initial Inspection

In [ ]:
# Load the UCI Mathematics student performance dataset.
df2 = pd.read_csv("student-mat.csv", sep=";")

In [ ]:
# Display the first few observations to inspect the dataset.
df2.head()

In [ ]:
df2.shape

In [ ]:
df2.info()

In [ ]:
df2.columns.tolist()

In [ ]:
df2.isnull().sum().sum()

## 3.2 Data Preparation

In [ ]:
# Exclude G1 and G2 from the main experiment so that final grade
# prediction is based on demographic, social, behavioural and
# other available characteristics.
# Use G3 as the prediction target.
X2 = df2.drop(columns=["G1", "G2", "G3"])
y2 = df2["G3"]

In [ ]:
print("Features:", X2.shape)
print("Target:", y2.shape)

In [ ]:
Features: (395, 30)
Target: (395,)

In [ ]:
# Separate categorical and numerical predictors for preprocessing.
categorical_cols2 = X2.select_dtypes(include=["object"]).columns.tolist()
numerical_cols2 = X2.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categorical columns:", categorical_cols2)
print("\nNumerical columns:", numerical_cols2)

## 3.3 Train-Test Split and Preprocessing

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2,
    y2,
    test_size=0.2,
    random_state=42
)

print("Training set:", X2_train.shape)
print("Testing set:", X2_test.shape)

In [ ]:
preprocessor2 = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols2),
        ("numerical", StandardScaler(), numerical_cols2)
    ]
)

## 3.4 Random Forest Regression

In [ ]:
# Create a pipeline that preprocesses the predictors and trains
# a Random Forest regression model.
rf_regressor = Pipeline(
    steps=[
        ("preprocessor", preprocessor2),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

rf_regressor.fit(X2_train, y2_train)

In [ ]:
# Generate final-grade predictions for the test set.
y2_pred_rf = rf_regressor.predict(X2_test)

In [ ]:
# Calculate MAE, RMSE and R-squared to evaluate Random Forest
# regression performance.
rf_mae = mean_absolute_error(y2_test, y2_pred_rf)
rf_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_rf))
rf_r2 = r2_score(y2_test, y2_pred_rf)

print("Random Forest Regression Results")
print("---------------------------------")
print(f"MAE : {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²  : {rf_r2:.4f}")

## 3.5 KNN Regression

In [ ]:
# Create a pipeline that preprocesses the predictors and applies
# K-Nearest Neighbours regression.
knn_regressor = Pipeline(
    steps=[
        ("preprocessor", preprocessor2),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]
)

knn_regressor.fit(X2_train, y2_train)

In [ ]:
# Generate final-grade predictions for the test set using KNN.
y2_pred_knn = knn_regressor.predict(X2_test)

In [ ]:
# Calculate MAE, RMSE and R-squared to evaluate KNN regression
# performance.
knn_mae = mean_absolute_error(y2_test, y2_pred_knn)
knn_rmse = np.sqrt(mean_squared_error(y2_test, y2_pred_knn))
knn_r2 = r2_score(y2_test, y2_pred_knn)

print("KNN Regression Results")
print("----------------------")
print(f"MAE : {knn_mae:.4f}")
print(f"RMSE: {knn_rmse:.4f}")
print(f"R²  : {knn_r2:.4f}")

## 3.6 Model Comparison

In [ ]:
# Compare Random Forest and KNN using the selected regression
# evaluation metrics.

model_comparison2 = pd.DataFrame({
    "Model": ["Random Forest", "KNN"],
    "MAE": [rf_mae, knn_mae],
    "RMSE": [rf_rmse, knn_rmse],
    "R²": [rf_r2, knn_r2]
})

model_comparison2

## 4 Additional Experiment: Including Previous Grades

A second experiment was conducted using G1 and G2 as predictors of G3.
This allows comparison between prediction based on background and
behavioural characteristics alone and prediction when previous academic
performance is also available.

In [ ]:
# Prepare a second experiment that includes previous grades G1 and G2
# as predictors while using G3 as the target.
    X2_with_grades = df2.drop(columns=["G3"])
y2_with_grades = df2["G3"]


In [ ]:
# Check the dimensions of the predictors and target for the
# additional experiment.
print("Features:", X2_with_grades.shape)
print("Target:", y2_with_grades.shape)

In [ ]:
# Identify categorical and numerical predictors for preprocessing.
categorical_cols2g = X2_with_grades.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols2g = X2_with_grades.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

In [ ]:
# Split the data into training and testing sets for the additional
# experiment.
X2g_train, X2g_test, y2g_train, y2g_test = train_test_split(
    X2_with_grades,
    y2_with_grades,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Preprocess categorical and numerical predictors for the
# additional Random Forest experiment.
preprocessor2g = ColumnTransformer(
    transformers=[
        ("categorical",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_cols2g),
        ("numerical",
         StandardScaler(),
         numerical_cols2g)
    ]
)

In [ ]:
# Train a Random Forest regression model using the predictors
# including previous grades G1 and G2.
rf_regressor_grades = Pipeline(
    steps=[
        ("preprocessor", preprocessor2g),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42
        ))
    ]
)

rf_regressor_grades.fit(X2g_train, y2g_train)

In [ ]:
y2g_pred_rf = rf_regressor_grades.predict(X2g_test)

In [ ]:
# Evaluate the additional Random Forest model using MAE, RMSE
# and R-squared.
rf_g_mae = mean_absolute_error(y2g_test, y2g_pred_rf)
rf_g_rmse = np.sqrt(mean_squared_error(y2g_test, y2g_pred_rf))
rf_g_r2 = r2_score(y2g_test, y2g_pred_rf)

print("Random Forest — Including G1 and G2")
print("------------------------------------")
print(f"MAE : {rf_g_mae:.4f}")
print(f"RMSE: {rf_g_rmse:.4f}")
print(f"R²  : {rf_g_r2:.4f}")

In [ ]:
#Model Comparison with mail model

In [ ]:
# Compare Random Forest performance before and after including
# previous academic grades G1 and G2.
g1_g2_comparison = pd.DataFrame({
    "Feature Setup": ["Without G1/G2", "With G1/G2"],
    "MAE": [rf_mae, rf_g_mae],
    "RMSE": [rf_rmse, rf_g_rmse],
    "R²": [rf_r2, rf_g_r2]
})

g1_g2_comparison

Including G1 and G2 substantially improved Random Forest performance. 
The R² increased from 0.3092 to 0.8026, while both MAE and RMSE decreased.
This indicates that previous academic performance provides substantial
predictive information about the final grade (G3).

# 4.1 Feature Importance

In [ ]:
# Extract the processed feature names and Random Forest importance
# scores for the UCI regression model.
feature_names2 = (
    rf_regressor.named_steps["preprocessor"]
    .get_feature_names_out()
)

importances2 = rf_regressor.named_steps["model"].feature_importances_

feature_importance2 = pd.DataFrame({
    "Feature": feature_names2,
    "Importance": importances2
}).sort_values("Importance", ascending=False)

feature_importance2.head(15)

In [ ]:
# Visualise the ten most important predictors of final grade
# identified by the Random Forest model.
top10_2 = feature_importance2.head(10).sort_values("Importance")

plt.figure(figsize=(10, 6))
plt.barh(top10_2["Feature"], top10_2["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Top 10 Predictive Features — UCI Mathematics")
plt.tight_layout()
plt.show()

# 5. Key Findings and Limitations

## Key Findings

### Dataset 1 — OpenML 43415

Random Forest achieved better classification performance than KNN, with an
accuracy of 77.08% compared with 71.88% for KNN. The Random Forest feature
importance results identified student engagement and attendance-related
variables as important predictive signals. In particular, raised hands,
visited resources, announcements viewed, discussion and student absence
days were among the strongest features.

### Dataset 2 — UCI Student Performance

Random Forest also outperformed KNN for the regression task. Random Forest
achieved an MAE of 2.9715, RMSE of 3.7637 and R² of 0.3092, whereas KNN
achieved an MAE of 3.6405, RMSE of 4.6430 and R² of -0.0513. Absences and
failures were the strongest features in the Random Forest model, followed
by factors such as health, going out and study time.

### Comparison Across the Datasets

The two datasets provide complementary insights. Both identify
attendance or absence-related information as an important predictive
signal, while the other important variables differ between the datasets.
The OpenML dataset emphasises classroom engagement and participation,
whereas the UCI dataset highlights absences, failures and behavioural
factors.

### Effect of Previous Grades

The additional UCI experiment showed that including G1 and G2 substantially
improved Random Forest performance. R² increased from 0.3092 without G1
and G2 to 0.8026 when they were included, while MAE decreased from 2.9715
to 1.1903 and RMSE decreased from 3.7637 to 2.0117. This indicates that
previous academic performance provides substantial predictive information
about the final grade.

## Limitations

Feature importance indicates which variables contributed to the model's
predictions but does not establish causal relationships. Therefore, the
identified factors should not be interpreted as directly causing changes
in student performance.

The datasets are also relatively small and represent specific educational
contexts, which may limit how well the findings generalise to other
student populations or education systems. In addition, the two datasets
use different target variables and modelling tasks, so their performance
metrics cannot be directly compared.

For the primary UCI analysis, G1 and G2 were excluded to examine prediction
using demographic, family, behavioural and educational characteristics
without previous grades. The additional experiment demonstrates that
including previous grades substantially increases predictive performance.